# Module 04 — Context Management: Keeping Knowledge Fresh

In this notebook you will learn:
- How to detect stale data in your knowledge base
- Strategies for updating and re-indexing documents
- How to handle conflicting or outdated information
- Building a simple automated update pipeline


## 1. The Staleness Problem

A RAG knowledge base is only as good as its data. Common problems:

| Problem | Example | Fix |
|---------|---------|-----|
| Outdated facts | Old product prices | Re-index on schedule |
| Deleted documents | Removed policy pages | Track deletions |
| Conflicting versions | Old + new version of a doc | Versioning strategy |
| Missing new content | New docs not indexed yet | Incremental indexing |


## 2. Document Metadata for Freshness Tracking

Always store metadata alongside your documents so you can detect and filter stale content.


In [ ]:
import chromadb
from datetime import datetime, timedelta
import hashlib

client = chromadb.Client()
collection = client.create_collection('managed_knowledge_base')

def document_hash(text: str) -> str:
    """Hash document content to detect changes."""
    return hashlib.md5(text.encode()).hexdigest()

def add_document(doc_id: str, text: str, source: str, ttl_days: int = 30):
    """Add a document with freshness metadata."""
    now = datetime.now()
    expires_at = now + timedelta(days=ttl_days)
    
    collection.upsert(
        ids=[doc_id],
        documents=[text],
        metadatas=[{
            'source': source,
            'indexed_at': now.isoformat(),
            'expires_at': expires_at.isoformat(),
            'content_hash': document_hash(text),
            'ttl_days': ttl_days
        }]
    )

# Add documents with different TTLs
add_document('policy_v1', 'Return policy: 30 days for all items.', 'policy_page', ttl_days=90)
add_document('pricing_q1', 'Product A costs $99. Product B costs $149.', 'pricing_page', ttl_days=7)
add_document('faq_001', 'To reset your password, click Forgot Password on the login page.', 'faq', ttl_days=180)

print(f'Documents indexed: {collection.count()}')

## 3. Detecting Stale Documents


In [ ]:
def find_stale_documents() -> list[dict]:
    """Find documents that have expired or are close to expiry."""
    all_docs = collection.get(include=['metadatas', 'documents'])
    now = datetime.now()
    stale = []
    
    for doc_id, doc_text, metadata in zip(
        all_docs['ids'], all_docs['documents'], all_docs['metadatas']
    ):
        expires_at = datetime.fromisoformat(metadata['expires_at'])
        days_until_expiry = (expires_at - now).days
        
        if days_until_expiry <= 0:
            status = 'EXPIRED'
        elif days_until_expiry <= 7:
            status = 'EXPIRING_SOON'
        else:
            status = 'FRESH'
        
        stale.append({
            'id': doc_id,
            'source': metadata['source'],
            'status': status,
            'days_until_expiry': days_until_expiry,
            'preview': doc_text[:60] + '...'
        })
    
    return sorted(stale, key=lambda x: x['days_until_expiry'])

stale_docs = find_stale_documents()
for doc in stale_docs:
    print(f"[{doc['status']:14s}] {doc['id']:15s} (expires in {doc['days_until_expiry']} days) — {doc['preview']}")

## 4. Update Strategy

When you have new content, use `upsert` — it updates the document if the ID exists, inserts if it doesn't.


In [ ]:
def update_document(doc_id: str, new_text: str, source: str, ttl_days: int = 30):
    """Update an existing document and reset its TTL."""
    # Check if content actually changed (avoid unnecessary re-indexing)
    existing = collection.get(ids=[doc_id], include=['metadatas'])
    
    if existing['ids']:
        old_hash = existing['metadatas'][0].get('content_hash')
        new_hash = document_hash(new_text)
        
        if old_hash == new_hash:
            print(f'{doc_id}: Content unchanged, only refreshing TTL')
            # Just update the metadata (reset expiry)
            existing_text = collection.get(ids=[doc_id], include=['documents'])['documents'][0]
            add_document(doc_id, existing_text, source, ttl_days)
            return
    
    print(f'{doc_id}: Content changed, re-indexing')
    add_document(doc_id, new_text, source, ttl_days)

# Simulate a pricing update
update_document(
    'pricing_q1',
    'Product A costs $89. Product B costs $129.',  # prices changed!
    'pricing_page',
    ttl_days=7
)

# Simulate a no-change refresh
update_document(
    'faq_001',
    'To reset your password, click Forgot Password on the login page.',
    'faq',
    ttl_days=180
)

## 5. Freshness-Aware Retrieval

Filter out expired documents at query time as a safety net.


In [ ]:
def fresh_retrieve(query: str, n_results: int = 3) -> list[dict]:
    """Retrieve documents, excluding expired ones."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results * 2,  # fetch extra in case some are filtered
        include=['documents', 'metadatas', 'distances']
    )
    
    now = datetime.now()
    fresh_results = []
    
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        expires_at = datetime.fromisoformat(meta['expires_at'])
        if expires_at > now:
            fresh_results.append({
                'document': doc,
                'source': meta['source'],
                'similarity_score': 1 - dist,
                'expires_at': meta['expires_at']
            })
        
        if len(fresh_results) == n_results:
            break
    
    return fresh_results

results = fresh_retrieve('What is the price of Product A?')
for r in results:
    print(f"[{r['similarity_score']:.2f}] ({r['source']}) {r['document']}")

## 6. Building an Update Pipeline

In production, you'd schedule this to run automatically (e.g., daily via a cron job or Airflow).


In [ ]:
def run_update_pipeline(new_documents: list[dict]):
    """
    Simulate a scheduled update pipeline.
    
    In production, new_documents would come from:
    - Web scraping
    - Database queries
    - API calls to internal systems
    - File system watchers
    """
    print('=== Running Update Pipeline ===')
    
    # Step 1: Find stale documents
    stale = [d for d in find_stale_documents() if d['status'] in ('EXPIRED', 'EXPIRING_SOON')]
    print(f'Found {len(stale)} documents needing attention')
    
    # Step 2: Update with new content
    for doc in new_documents:
        update_document(
            doc_id=doc['id'],
            new_text=doc['text'],
            source=doc['source'],
            ttl_days=doc.get('ttl_days', 30)
        )
    
    # Step 3: Report
    print(f'\nPipeline complete. Total documents: {collection.count()}')

# Simulate running the pipeline with new data
run_update_pipeline([
    {'id': 'pricing_q2', 'text': 'Q2 pricing: Product A $85, Product B $125.', 'source': 'pricing_page', 'ttl_days': 7},
    {'id': 'policy_v2', 'text': 'Return policy updated: 60 days for all items, free returns.', 'source': 'policy_page', 'ttl_days': 90},
])

## Key Takeaways

1. **Always store metadata** — indexed_at, expires_at, source, content_hash
2. **Use content hashing** to avoid re-indexing unchanged documents
3. **Filter at retrieval time** as a safety net against stale data
4. **Automate updates** via scheduled pipelines
5. **TTL varies by content type** — pricing (short), policies (medium), general facts (long)

## Next Steps
Move on to `05_evaluation/` to learn how to measure whether your system is actually working well.
